In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from smmargins import Margins

rng = np.random.default_rng(42)
N = 6_000
df_did = pd.DataFrame({
    "group":      rng.choice(["A", "B"], N, p=[0.55, 0.45]),
    "preexist_Y": rng.integers(0, 2, N),
    "age":        rng.normal(55, 15, N).clip(18, 95),
    "female":     rng.integers(0, 2, N),
})
eta = (-3.5 + 0.04*df_did["age"] - 0.3*df_did["female"]
       + 0.5*(df_did["group"]=="B") + 1.1*df_did["preexist_Y"]
       + 0.8*(df_did["group"]=="B")*df_did["preexist_Y"])
df_did["condition_X"] = (rng.uniform(0,1,N) < 1/(1+np.exp(-eta))).astype(int)

fit_did = smf.logit("condition_X ~ C(group) + preexist_Y + C(group):preexist_Y + age + female",
                    data=df_did).fit(disp=False)
M_did = Margins(fit_did)

In [2]:
did = M_did.did("group", "preexist_Y",
                group_levels=["A", "B"],
                condition_levels=[0, 1])

In [3]:
did.cells.summary()

,prediction,std err,z,P>|z|,[95% Conf.,Interval]
"group=A, preexist_Y=0",0.198022,0.009412,21.038469,2.916422e-98,0.179574,0.216470
"group=A, preexist_Y=1",0.410620,0.011475,35.784808,1.903121e-280,0.388130,0.433110
"group=B, preexist_Y=0",0.299717,0.011959,25.062174,1.286254e-138,0.276278,0.323156
"group=B, preexist_Y=1",0.716206,0.011964,59.865312,0.000000e+00,0.692758,0.739654


In [4]:
did.simple_effects.summary()

,simple effect,std err,z,P>|z|,[95% Conf.,Interval]
group: B vs A | preexist_Y=0,0.101695,0.015219,6.682253,2.352960e-11,0.071867,0.131523
group: B vs A | preexist_Y=1,0.305586,0.016580,18.431265,7.373879e-76,0.273090,0.338082


In [5]:
did.did.summary()

,DiD,std err,z,P>|z|,[95% Conf.,Interval]
DiD: group(B-A) × preexist_Y(1-0),0.203891,0.022504,9.06029,1.301042e-19,0.159784,0.247998


In [6]:
did_profile = M_did.did("group", "preexist_Y",
                        group_levels=["A", "B"],
                        condition_levels=[0, 1],
                        atexog={"age": 60, "female": 0})

In [7]:
did_profile.cells.summary()

,prediction,std err,z,P>|z|,[95% Conf.,Interval]
"group=A, preexist_Y=0",0.241826,0.012700,19.042046,7.648041e-81,0.216935,0.266717
"group=A, preexist_Y=1",0.496332,0.015304,32.430587,1.017384e-230,0.466336,0.526328
"group=B, preexist_Y=0",0.366280,0.015858,23.097864,4.864630e-118,0.335199,0.397360
"group=B, preexist_Y=1",0.801633,0.011731,68.332726,0.000000e+00,0.778640,0.824626


In [8]:
did_profile.did.summary()

,DiD,std err,z,P>|z|,[95% Conf.,Interval]
DiD: group(B-A) × preexist_Y(1-0),0.180847,0.02517,7.185068,6.717382e-13,0.131515,0.230179
